In [ ]:
import pandas as pd
import numpy as np

Combine what 

average visitation wait between visits 
based on diagnosis 
one on cancer one one 
different columns by region 

In [ ]:
encounters = pd.read_csv("encounters.csv")
patients = pd.read_csv('patients.csv')
diagnosis = pd.read_csv('diagnosis.csv')
departments = pd.read_csv('departments.csv')
providers = pd.read_csv('providers.csv')
social_determinants = pd.read_csv('social_determinants.csv')
tigercensuscodes = pd.read_csv('tigercensuscodes.csv')

In [ ]:
encounters.head()

In [ ]:
patients.head()

In [ ]:
diagnosis.head()

In [ ]:
departments.head()

In [ ]:
providers
# look into this 
# providers is a person working within a department who provides heath care, such as  a technician, physician, or nurse practitioner.
# this file provides info about where this person works + their medical specialty
# ClinitianTitle - official Licensures of the care provider (i.e. Registered Nurse, Medical Doctor, etc.)	
# DurableKey - Links to patients file via ProviderDurableKey	
# OfficeAddress	Address of provider's primary work location	
# OfficePostalCode	Postal (Zip) Code of provider's primary work location	
# OfficeCity	City of provider's primary work location	
# PrimaryDepartment	The primary location  at which the provider sees  most of their patients.	
# PrimarySpecialty	The provider's specialty. What kind of patients and conditions do they see.	
# Type	e.g. physician, nurse practitioner, therapist, the field of the provider. A slightly higher level of description than Clinicial Title, which provides more detail. Provides general description of ClinicianTitle.

# explore null values in providers.csv
providers.isnull().sum()	
# large number of missing values for ClinicianTitle, OfficeAddress, OfficePostalCode, OfficeCity, and PrimarySpecialty.
# What to do from here?
# want to look at the specific providers with missing values and see if there are any patterns in the data that could help us fill in the missing values.

providers[providers['ClinicianTitle'].isnull()]
# unsure if there are any patterns in the data that could help us fill in the missing values for ClinicianTitle.


providers[providers['OfficeAddress'].isnull()]
# i see a lot of nursing students with missing values for OfficeAddress, OfficePostalCode, and OfficeCity. This could be because they are not yet assigned to a specific location or department within the healthcare system.

providers[providers['PrimarySpecialty'].isnull()]
providers[providers['PrimarySpecialty'].isnull()]['Type'].unique()
# for a lot of these, it looks like they are assistants, nurses, or nursing students, as well as therapists or counselors. This could be because they do not have a specific medical specialty, or because their specialty is not relevant to the care they provide.

providers.isnull().sum()	



In [ ]:
social_determinants.head()

# merge with provided supporting dataset to remove null values 
# has a category for transportation 

# social determinants 

In [ ]:
patients.head() 

# want to see which values in patients are *Unspecified for their census block 
patients[patients['CensusBlockGroupFipsCode'].isnull()]

In [ ]:
# Our high-level game plan 
# choosing to focus on transportation / geographic access as our social determinant of health for this project, as it is a common barrier to healthcare access and can have a significant impact on patient outcomes.
# also choosing 2 core diagnosis families / types to focus one: one longer term (cancer) and one shorter term (fractures). This will allow us to compare and contrast the impact of transportation barriers on different types of health conditions, and to identify any unique challenges or opportunities for improving access to care for patients with these conditions.
# for each diagnosis family, we want to look at average visitations needed per patient and average staying time, and segment by region to see if there are any geographic patterns in healthcare utilization and outcomes for patients with transportation barriers.


# Fractures
diagnosis['GroupName'].str.contains('fracture', case=False).mean()

# merge encounters and diagnosis datasets
encounters_diagnosis = pd.merge(encounters, diagnosis, left_on='PrimaryDiagnosisKey', right_on='DiagnosisKey')



In [ ]:
encounters_diagnosis.head()

# generate column names of encounters_diagnosis
encounters_diagnosis.columns

In [ ]:
# filter for fractures
fractures = encounters_diagnosis[encounters_diagnosis['GroupName'].str.contains('fracture', case=False)]
fractures

# look at all GroupNames for fractures
fractures['GroupName'].unique()

# Filter out 'osteoporosis without current pathological fracture' from dataset
fractures = fractures[~fractures['GroupName'].str.contains('osteoporosis without current pathological fracture', case=False)]
fractures['GroupName'].unique()

In [ ]:
# calculate average visitations needed per patient for fractures
fractures_visitations = fractures.groupby('PatientDurableKey')['EncounterKey'].nunique().mean()
print(f"Average visitations needed per patient for fractures: {fractures_visitations}")


In [ ]:
# Take the patient.csv dataset and encounters.csv dataset and merge them together on the PatientDurableKey column. This will allow us to have access to the patient's census block group information. 

# start with patients.csv 
patients
patients.columns 
# key columns: CensusBlockGroupFipsCode, DurableKey

patients = patients[
    patients["CensusBlockGroupFipsCode"].notna() &
    (patients["CensusBlockGroupFipsCode"] != "*Unspecified")
] # removing patients with null or unspecified census block group information - unsure whether or not to do this 

tigercensuscodes.head()

tigercensuscodes["GEOID"] = tigercensuscodes["GEOID"].astype(str)
tigercensuscodes["CountyFips"] = tigercensuscodes["GEOID"].str[:5]


tiger_geo = tigercensuscodes[[
    "GEOID",
    "CountyFips",
    "CENTLAT",
    "CENTLON",
    "PopulationValue"
]]


patients_geo = patients.merge(
    tiger_geo,
    left_on="CensusBlockGroupFipsCode",
    right_on="GEOID",
    how="left"
)


encounters_geo = encounters.merge(
    patients_geo[
        [
            "DurableKey",          # patient identifier
            "CountyFips",
            "CENTLAT",
            "CENTLON",
            "PopulationValue"
        ]
    ],
    left_on="PatientDurableKey",   # from encounters
    right_on="DurableKey",         # from patients
    how='left'
)


In [ ]:
encounters_geo.head()

In [ ]:
# % of encounters missing county
missing_county_pct = encounters_geo["CountyFips"].isna().mean()
print(f"Missing county: {missing_county_pct:.2%}")

# Ensure each patient maps to exactly one county
patients_geo.groupby("DurableKey")["CountyFips"].nunique().describe()


In [ ]:
county_mapping = pd.DataFrame({
    "CountyCode": [
        "001","003","005","007","009","011","013","015","017","019","021","023","025","027",
        "029","031","033","035","037","039","041","043","045","047","049","051","053","055",
        "057","059","061","063","065","067","069","071","073","075","077","079","081","083",
        "085","087","089","091","093","095","097","099","101","103","105","107","109","111",
        "113","115","117","119","121","123","125","127","129","131","133","135","137","139",
        "141","143","145","147","149","151","153","155","157","159","161","163","165","167",
        "169","171","173","175","177","179","181","183","185","187","189","191","193","195",
        "197","199","201","203","205","207","209"
    ],
    "CountyName": [
        "Allen","Anderson","Atchison","Barber","Barton","Bourbon","Brown","Butler","Chase",
        "Chautauqua","Cherokee","Cheyenne","Clark","Clay","Cloud","Coffey","Comanche","Cowley",
        "Crawford","Decatur","Dickinson","Doniphan","Douglas","Edwards","Elk","Ellis","Ellsworth",
        "Finney","Ford","Franklin","Geary","Gove","Graham","Grant","Gray","Greeley","Greenwood",
        "Hamilton","Harper","Harvey","Haskell","Hodgeman","Jackson","Jefferson","Jewell",
        "Johnson","Kearny","Kingman","Kiowa","Labette","Lane","Leavenworth","Lincoln","Linn",
        "Logan","Lyon","McPherson","Marion","Marshall","Meade","Miami","Mitchell","Montgomery",
        "Morris","Morton","Nemaha","Neosho","Ness","Norton","Osage","Osborne","Ottawa","Pawnee",
        "Phillips","Pottawatomie","Pratt","Rawlins","Reno","Republic","Rice","Riley","Rooks",
        "Rush","Russell","Saline","Scott","Sedgwick","Seward","Shawnee","Sheridan","Sherman",
        "Smith","Stafford","Stanton","Stevens","Sumner","Thomas","Trego","Wabaunsee","Wallace",
        "Washington","Wichita","Wilson","Woodson","Wyandotte"
    ]
})

In [ ]:
encounters_geo["CountyCode"] = encounters_geo["CountyFips"].str[-3:]


encounters_geo = encounters_geo.merge(
    county_mapping,
    on="CountyCode",
    how="left"
)

encounters_geo.head()

In [ ]:
encounters_geo['CountyName'].unique()

In [ ]:
print(encounters_geo['CountyName'].isna().mean())
encounters_geo = encounters_geo.dropna(subset=['CountyName'])


In [ ]:
region_map = {
    "Shawnee": "Northeast Kansas",
    "Jefferson": "Northeast Kansas",
    "Riley": "Northeast Kansas",
    "Lyon": "Northeast Kansas",
    "Douglas": "Northeast Kansas",
    "Osage": "Northeast Kansas",
    "Nemaha": "Northeast Kansas",
    "Wabaunsee": "Northeast Kansas",
    "Pottawatomie": "Northeast Kansas",
    "Jackson": "Northeast Kansas",
    "Johnson": "Northeast Kansas",
    "Franklin": "Northeast Kansas",
    "Coffey": "Northeast Kansas",
    "Chase": "Northeast Kansas",
    "Atchison": "Northeast Kansas",
    "Geary": "Northeast Kansas",
    "Wyandotte": "Northeast Kansas",
    "Marshall": "Northeast Kansas",
    "Morris": "Northeast Kansas",
    "Leavenworth": "Northeast Kansas",
    "Brown": "Northeast Kansas",
    "Doniphan": "Northeast Kansas",
    "Miami": "Northeast Kansas",

    "Cheyenne": "Northwest Kansas",
    "Rawlins": "Northwest Kansas",
    "Decatur": "Northwest Kansas",
    "Norton": "Northwest Kansas",
    "Phillips": "Northwest Kansas",
    "Sherman": "Northwest Kansas",
    "Thomas": "Northwest Kansas",
    "Sheridan": "Northwest Kansas",
    "Graham": "Northwest Kansas",
    "Rooks": "Northwest Kansas",
    "Wallace": "Northwest Kansas",
    "Logan": "Northwest Kansas",
    "Gove": "Northwest Kansas",
    "Trego": "Northwest Kansas",
    "Ellis": "Northwest Kansas",

    "Smith": "North Central Kansas",
    "Jewell": "North Central Kansas",
    "Republic": "North Central Kansas",
    "Washington": "North Central Kansas",
    "Osborne": "North Central Kansas",
    "Mitchell": "North Central Kansas",
    "Cloud": "North Central Kansas",
    "Clay": "North Central Kansas",
    "Lincoln": "North Central Kansas",
    "Ottawa": "North Central Kansas",
    "Russell": "North Central Kansas",
    "Saline": "North Central Kansas",
    "Ellsworth": "North Central Kansas",
    "Dickinson": "North Central Kansas",

    "Ford": "Southwest Kansas",
    "Finney": "Southwest Kansas",
    "Scott": "Southwest Kansas",
    "Grant": "Southwest Kansas",
    "Pratt": "Southwest Kansas",
    "Edwards": "Southwest Kansas",
    "Barber": "Southwest Kansas",
    "Ness": "Southwest Kansas",

    "Sedgwick": "South Central Kansas",
    "McPherson": "South Central Kansas",
    "Harvey": "South Central Kansas",
    "Cowley": "South Central Kansas",
    "Butler": "South Central Kansas",
    "Pawnee": "South Central Kansas",
    "Reno": "South Central Kansas",
    "Rush": "South Central Kansas",
    "Barton": "South Central Kansas",
    "Rice": "South Central Kansas",
    "Harper": "South Central Kansas",
    "Stafford": "South Central Kansas",

    "Woodson": "Southeast Kansas",
    "Greenwood": "Southeast Kansas",
    "Neosho": "Southeast Kansas",
    "Allen": "Southeast Kansas",
    "Wilson": "Southeast Kansas",
    "Labette": "Southeast Kansas",
    "Montgomery": "Southeast Kansas",
    "Crawford": "Southeast Kansas",
    "Chautauqua": "Southeast Kansas",
    "Bourbon": "Southeast Kansas",
    "Elk": "Southeast Kansas",
    "Linn": "Southeast Kansas",
    "Anderson": "Southeast Kansas"
}

encounters_geo['Region'] = encounters_geo['CountyName'].map(region_map)

In [ ]:
encounters_geo.head()

In [ ]:
df = pd.merge(encounters, diagnosis, left_on='PrimaryDiagnosisKey', right_on='DiagnosisKey')
df['CANCER'] = df['GroupName'].str.contains('malignant', case=False, na=False).astype(int)
cancer_df = df.loc[df["CANCER"] == 1]
cancer_df["Date"] = pd.to_datetime(cancer_df["Date"], format='mixed')

In [ ]:
# df should be a sub-dataframe of encounters with all encounters involving the relevant diagnoses
# in your input df, make sure you change the "Date" column to datetime objects
def time_between_visits(df):
    results = []
    for patient_key in df["PatientDurableKey"].unique():
        patient_df = df.loc[df["PatientDurableKey"] == patient_key].sort_values(by = "Date", ascending = True)
        
        if len(patient_df) == 1:
            avg_wait = pd.to_timedelta(0, unit = 's')
            
        else: 
            time_btwn = []
            for i in range(0, len(patient_df) - 1):
                time_btwn.append(patient_df.iloc[i+1, 0] - patient_df.iloc[i, 0])
            avg_wait = np.mean(time_btwn)

        results.append({"Patient Key": patient_key, "Avg Wait Time": avg_wait})

    return pd.DataFrame(results)

In [ ]:
cancer_waittime_df = time_between_visits(cancer_df)

In [ ]:
cancer_waittime_df.head()

i want to visualize bar charts using shiny using python. y axis is average wait between visits, x axis is region. 3 bar charts, one on cancer, one on fracture, one on depression.

In [ ]:
from shiny import App, render, ui
import plotly.express as px

app_ui = ui.page_fluid(
    ui.h2("Cancer: Average Wait Time Between Visits by Region"),
    ui.output_plot("plot_cancer")
)

def server(input, output, session):

    @output
    @render.plot
    def plot_cancer():
        df_plot = (
            cancer_waittime_df
            .groupby("Region")["AvgWaitDays"]
            .mean()
            .reset_index()
        )

        fig = px.bar(
            df_plot,
            x="Region",
            y="AvgWaitDays",
            title="Cancer: Average Wait Time (Days)",
            labels={"AvgWaitDays": "Avg Wait (Days)", "Region": "Region"},
            text_auto=True
        )

        fig.update_layout(
            xaxis_tickangle=-45,
            title_x=0.5
        )

        return fig

app = App(app_ui, server)
